# Trabalho – Chatbot Baseado em Regras – Parte 02

**Nome: Thais Silva Ferreira**

**Disciplina:** Chatbots e Assistentes Virtuais  
**Curso:** Ciência de Dados e Inteligência Artificial Aplicada ao Mercado Financeiro  
**Professor:** Ricardo Brito Alves

## Objetivo

Implementar um chatbot baseado exclusivamente em regras, seguindo fluxos conversacionais pré-definidos, sem utilização de IA generativa. O chatbot foi desenvolvido para um contexto de atendimento bancário e possui três fluxos principais: consulta de fatura, bloqueio de cartão e solicitação de aumento de limite.


## Funcionalidades implementadas

- Menu inicial funcional.
- Três fluxos conversacionais completos.
- Perguntas fechadas e menus.
- Tratamento de entradas inválidas por fallback.
- Comandos `menu`, `voltar` e `sair`.
- Encerramento claro da conversa.
- Registro das interações em arquivos de log.


In [ ]:
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path


def normalize(text: str) -> str:
    return " ".join(text.strip().lower().split())


@dataclass
class BotState:
    running: bool = True
    current_flow: str = "menu"


## Registro das conversas

Cada interação pode ser registrada automaticamente em um arquivo de texto na pasta `logs`.


In [ ]:
class ChatLogger:
    def __init__(self, base_dir: str = "logs") -> None:
        Path(base_dir).mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.path = Path(base_dir) / f"chatbot_log_{stamp}.txt"
        self._write_header()

    def _write_header(self) -> None:
        header = (
            "TP1 - Chatbot Baseado em Regras (Log de Conversa)\n"
            f"Data/Hora início: {datetime.now().isoformat(sep=' ', timespec='seconds')}\n"
            "Formato: [timestamp] (flow=...) USER: ... | BOT: ...\n"
            "------------------------------------------------------------\n"
        )
        self.path.write_text(header, encoding="utf-8")

    def log_turn(self, flow: str, user_msg: str, bot_msg: str) -> None:
        ts = datetime.now().isoformat(sep=" ", timespec="seconds")
        line = f"[{ts}] (flow={flow}) USER: {user_msg} | BOT: {bot_msg}\n"
        with self.path.open("a", encoding="utf-8") as file:
            file.write(line)


## Menu inicial e mensagens gerais


In [ ]:
def show_menu() -> str:
    return (
        "\n=== MENU PRINCIPAL ===\n"
        "1) Consultar fatura\n"
        "2) Bloquear cartão\n"
        "3) Aumentar limite\n"
        "Digite 1, 2, 3 ou 'sair'.\n"
        "Você também pode digitar 'menu' ou 'voltar' a qualquer momento.\n"
    )


def fallback() -> str:
    return (
        "Não entendi sua resposta. Digite uma das opções disponíveis "
        "ou 'menu' para voltar ao menu principal."
    )


def end_message() -> str:
    return "\nPosso ajudar em mais algo? Digite 'menu' para voltar ao menu ou 'sair' para encerrar."


## Fluxo 1 – Consultar fatura

O usuário pode consultar o valor atual, o vencimento e as formas de pagamento da fatura.


In [ ]:
def flow_fatura(state: BotState, msg: str) -> str:
    msg = normalize(msg)

    if msg in ("menu", "voltar"):
        state.current_flow = "menu"
        return show_menu()

    if msg == "sair":
        state.running = False
        return "Encerrando a conversa. Obrigado por utilizar nosso atendimento!"

    if msg in ("1", "valor", "valor atual"):
        return (
            "Sua fatura atual está em aberto. Para consultar o valor, "
            "acesse: App > Cartões > Fatura."
            + end_message()
        )

    if msg in ("2", "vencimento"):
        return (
            "O vencimento da sua fatura pode ser consultado em: "
            "App > Cartões > Fatura > Detalhes."
            + end_message()
        )

    if msg in ("3", "pagar", "pagamento"):
        return (
            "Para realizar o pagamento, acesse: App > Cartões > Fatura > Pagar. "
            "Você pode utilizar PIX ou boleto."
            + end_message()
        )

    return (
        "\n[FATURA]\n"
        "Escolha uma opção:\n"
        "1) Valor atual\n"
        "2) Vencimento\n"
        "3) Como pagar\n"
        "Ou digite 'menu' para voltar.\n"
    )


## Fluxo 2 – Bloquear cartão

O chatbot solicita uma confirmação fechada antes de orientar o usuário sobre o bloqueio.


In [ ]:
def flow_bloqueio(state: BotState, msg: str) -> str:
    msg = normalize(msg)

    if msg in ("menu", "voltar"):
        state.current_flow = "menu"
        return show_menu()

    if msg == "sair":
        state.running = False
        return "Encerrando a conversa. Obrigado por utilizar nosso atendimento!"

    if msg in ("sim", "s", "confirmo"):
        return (
            "Solicitação confirmada. Para bloquear o cartão, siga o caminho:\n"
            "App > Cartões > Segurança > Bloquear cartão.\n"
            "Em caso de perda ou roubo, também é recomendado registrar uma ocorrência."
            + end_message()
        )

    if msg in ("nao", "não", "n", "cancelar"):
        return "Tudo bem. O bloqueio foi cancelado." + end_message()

    return (
        "\n[BLOQUEAR CARTÃO]\n"
        "Você confirma que deseja bloquear o cartão agora?\n"
        "Digite 'sim' para confirmar ou 'não' para cancelar.\n"
        "Ou digite 'menu' para voltar.\n"
    )


## Fluxo 3 – Aumentar limite

O usuário escolhe entre solicitar um aumento temporário ou permanente.


In [ ]:
def flow_limite(state: BotState, msg: str) -> str:
    msg = normalize(msg)

    if msg in ("menu", "voltar"):
        state.current_flow = "menu"
        return show_menu()

    if msg == "sair":
        state.running = False
        return "Encerrando a conversa. Obrigado por utilizar nosso atendimento!"

    if msg in ("1", "temporario", "temporário"):
        return (
            "A solicitação de aumento temporário depende da análise do perfil e histórico.\n"
            "Acesse: App > Cartões > Limite > Solicitar aumento temporário."
            + end_message()
        )

    if msg in ("2", "permanente"):
        return (
            "A solicitação de aumento permanente pode exigir atualização de renda "
            "e análise do perfil financeiro.\n"
            "Acesse: App > Perfil > Renda > Atualizar e depois "
            "Cartões > Limite > Solicitar aumento."
            + end_message()
        )

    return (
        "\n[AUMENTAR LIMITE]\n"
        "Qual tipo de aumento você deseja?\n"
        "1) Temporário\n"
        "2) Permanente\n"
        "Ou digite 'menu' para voltar.\n"
    )


## Roteamento por regras


In [ ]:
def route_to_flow(state: BotState, msg: str) -> str:
    msg_norm = normalize(msg)

    if msg_norm == "sair":
        state.running = False
        return "Encerrando a conversa. Obrigado por utilizar nosso atendimento!"

    if msg_norm in ("menu", "voltar"):
        state.current_flow = "menu"
        return show_menu()

    if state.current_flow == "menu":
        if msg_norm in ("1", "fatura"):
            state.current_flow = "fatura"
            return flow_fatura(state, "")

        if msg_norm in ("2", "bloquear", "bloqueio", "bloquear cartao", "bloquear cartão"):
            state.current_flow = "bloqueio"
            return flow_bloqueio(state, "")

        if msg_norm in ("3", "limite", "aumentar limite"):
            state.current_flow = "limite"
            return flow_limite(state, "")

        return fallback()

    if state.current_flow == "fatura":
        return flow_fatura(state, msg_norm)

    if state.current_flow == "bloqueio":
        return flow_bloqueio(state, msg_norm)

    if state.current_flow == "limite":
        return flow_limite(state, msg_norm)

    state.current_flow = "menu"
    return show_menu()


## Execução interativa

Descomente a última linha para conversar com o chatbot diretamente no notebook.


In [ ]:
def executar_chatbot():
    state = BotState()
    logger = ChatLogger()

    print("Chatbot Bancário - Atendimento Baseado em Regras")
    print(show_menu())

    while state.running:
        user_msg = input("> ")
        bot_reply = route_to_flow(state, user_msg)
        print(bot_reply)

        logger.log_turn(
            flow=state.current_flow,
            user_msg=user_msg,
            bot_msg=bot_reply.replace("\n", "\\n")
        )

    print(f"\nLog salvo em: {logger.path.resolve()}")


# executar_chatbot()


# Testes obrigatórios

Os testes abaixo demonstram três conversas felizes, três entradas inválidas acionando o fallback, uso do comando `menu` no meio da conversa e encerramento correto.


In [ ]:
def testar_conversa(nome, entradas):
    state = BotState()

    print("\n" + "=" * 60)
    print(nome)
    print("=" * 60)
    print("BOT:", show_menu())

    for entrada in entradas:
        if not state.running:
            break

        resposta = route_to_flow(state, entrada)
        print(f"USUÁRIO: {entrada}")
        print(f"BOT: {resposta}")


### Teste 1 – Conversa feliz: consultar fatura


In [ ]:
testar_conversa(
    "TESTE 1 - CONSULTA DE FATURA",
    ["1", "1", "sair"]
)


### Teste 2 – Conversa feliz: bloquear cartão


In [ ]:
testar_conversa(
    "TESTE 2 - BLOQUEIO DE CARTÃO",
    ["2", "sim", "sair"]
)


### Teste 3 – Conversa feliz: aumentar limite


In [ ]:
testar_conversa(
    "TESTE 3 - AUMENTO DE LIMITE",
    ["3", "2", "sair"]
)


### Teste 4 – Três entradas inválidas acionando o fallback


In [ ]:
testar_conversa(
    "TESTE 4 - FALLBACK",
    ["abc", "99", "quero falar com alguém", "sair"]
)


### Teste 5 – Uso de `menu` no meio da conversa


In [ ]:
testar_conversa(
    "TESTE 5 - COMANDO MENU",
    ["1", "menu", "3", "1", "sair"]
)


### Teste 6 – Encerramento correto


In [ ]:
testar_conversa(
    "TESTE 6 - ENCERRAMENTO",
    ["sair"]
)


## Conclusão

A implementação do chatbot foi estruturada estritamente sobre regras condicionais e fluxos conversacionais pré-definidos. A arquitetura contempla menu principal, três rotas completas orientadas por perguntas fechadas, mecanismo de fallback para entradas inválidas, além de navegação para retorno ao menu e encerramento adequado da sessão. A etapa de testes validou a conformidade da solução com todos os requisitos obrigatórios especificados.
